### New Data

In [6]:
from __future__ import annotations

import os
import re
import shutil
import numpy as np
import pandas as pd
from ase import Atoms
from ase.io import read, write
from tqdm import tqdm
from sklearn.neighbors import NearestNeighbors

# ================= GRACEFUL NUMBA JIT DECORATOR =================
try:
    from numba import njit
except ImportError:
    # Fallback decorator if Numba is not installed
    def njit(*args, **kwargs):
        if len(args) == 1 and len(kwargs) == 0 and callable(args[0]):
            return args[0]
        def decorator(func):
            return func
        return decorator

# ================= JIT COMPRESSED MATH OPERATIONS (GIL Released) =================
# nogil=True allows these mathematical heavy loops to run in parallel using ThreadPoolExecutor
@njit(cache=True, fastmath=True, nogil=True)
def check_clash_pbc_jit(frac_orig, frac_new, cell, pbc, threshold_matrix):
    """
    Checks for atomic clashes between two fractional coordinate sets.
    Loops through pairs and exits early at the first clash (saves massive CPU cycles).
    """
    n_orig = frac_orig.shape[0]
    n_new = frac_new.shape[0]
    
    for i in range(n_orig):
        for j in range(n_new):
            dx = frac_orig[i, 0] - frac_new[j, 0]
            dy = frac_orig[i, 1] - frac_new[j, 1]
            dz = frac_orig[i, 2] - frac_new[j, 2]
            
            # Minimum image convention in fractional space
            if pbc[0]:
                dx -= np.round(dx)
            if pbc[1]:
                dy -= np.round(dy)
            if pbc[2]:
                dz -= np.round(dz)
                
            # Convert difference vector to Cartesian space
            cx = dx * cell[0, 0] + dy * cell[1, 0] + dz * cell[2, 0]
            cy = dx * cell[0, 1] + dy * cell[1, 1] + dz * cell[2, 1]
            cz = dx * cell[0, 2] + dy * cell[1, 2] + dz * cell[2, 2]
            
            dist = np.sqrt(cx*cx + cy*cy + cz*cz)
            if dist < threshold_matrix[i, j] - 0.05:
                return True
    return False

@njit(cache=True, fastmath=True, nogil=True)
def get_min_dist_jit(frac_orig, frac_new, cell, pbc):
    """
    Calculates the exact PBC-aware minimum contact distance between two layers.
    Optimized to compute distances without allocating memory arrays.
    """
    n_orig = frac_orig.shape[0]
    n_new = frac_new.shape[0]
    min_dist = 9999.0
    
    for i in range(n_orig):
        for j in range(n_new):
            dx = frac_orig[i, 0] - frac_new[j, 0]
            dy = frac_orig[i, 1] - frac_new[j, 1]
            dz = frac_orig[i, 2] - frac_new[j, 2]
            
            if pbc[0]:
                dx -= np.round(dx)
            if pbc[1]:
                dy -= np.round(dy)
            if pbc[2]:
                dz -= np.round(dz)
                
            cx = dx * cell[0, 0] + dy * cell[1, 0] + dz * cell[2, 0]
            cy = dx * cell[0, 1] + dy * cell[1, 1] + dz * cell[2, 1]
            cz = dx * cell[0, 2] + dy * cell[1, 2] + dz * cell[2, 2]
            
            dist = np.sqrt(cx*cx + cy*cy + cz*cz)
            if dist < min_dist:
                min_dist = dist
    return min_dist

# ================= AUTOMATIC PATH DETECTION =================
current_dir = os.getcwd()
if os.path.basename(current_dir) == 'script':
    PROJECT_ROOT = os.path.dirname(current_dir)
else:
    PROJECT_ROOT = current_dir

ROOT_IN = os.path.join(PROJECT_ROOT, "negative_2_cifs")
TM_CSV = os.path.join(PROJECT_ROOT, "results", "cif_layer_displacements.csv")
OUT_ROOT = os.path.join(PROJECT_ROOT, "results", "3_layer")

# ================= GLOBAL CONSTANTS =================
X_CART = 9.35
N = 3
Y = np.arange(0, (N*8), 0.5)
Z = np.arange(0, (N*8), 0.5)

# Selection of angles for third layer rotation
ANGLES = np.array([80, 100, 200, 220, 260, 280, 320, 340])

Y_CUT = 0.5
Z_CUT = 0.8

VDW_RADII = {
    'C': 1.70,
    'H': 1.20,
    'O': 1.52
}

def adjust_fractional(frac: np.ndarray, y_cut=Y_CUT, z_cut=Z_CUT) -> np.ndarray:
    f = frac.copy()
    f[:, 1] = np.where(f[:, 1] > y_cut, f[:, 1] - 1.0, f[:, 1])
    f[:, 2] = np.where(f[:, 2] > z_cut, f[:, 2] - 1.0, f[:, 2])
    return f

def extract_r_from_path(path: str) -> int:
    norm = path.replace("\\", "/")
    m = re.search(r"/r(\d+)(?:/|$)", norm)
    if not m:
        raise ValueError(f"Could not find rXX in path: {path}")
    return int(m.group(1))

def extract_t_from_path(path: str) -> str:
    norm = path.replace("\\", "/")
    m = re.search(r"/t([0-9.]+)/(?:[^/]+)$", norm)
    if not m:
        raise ValueError(f"Could not find tX.X in path: {path}")
    return m.group(1)

def extract_angle_from_filename(path: str) -> float:
    fname = os.path.basename(path)
    m = re.search(r"_([0-9]+)\.cif$", fname)
    if not m:
        raise ValueError(f"Could not find _NNN.cif at end of {fname}")
    return float(m.group(1))

def rotate_about_x(points: np.ndarray, angle_deg: float, pivot: np.ndarray) -> np.ndarray:
    theta = np.deg2rad(angle_deg)
    R = np.array([
        [1.0,         0.0,          0.0],
        [0.0,  np.cos(theta), -np.sin(theta)],
        [0.0,  np.sin(theta),  np.cos(theta)],
    ])
    shifted = points - pivot[None, :]
    return shifted @ R.T + pivot[None, :]

def get_displacement_for_cif(cif_path, df_disp):
    cif_norm = cif_path.replace('\\', '/').split('/')
    key = '/'.join(cif_norm[-3:])  # matches 'rXX/tXX/tXX_angle.cif'
    for _, row in df_disp.iterrows():
        path_norm = row['file_path'].replace('\\', '/')
        if path_norm.endswith(key):
            return np.array([row['X'], row['Y'], row['Z']])
    return None

def select_grids_radius_prune(df_pts: pd.DataFrame, target_n: int = 100, radius: float = 4.0, seed: int | None = None) -> pd.DataFrame:
    pts = df_pts[["gx", "gy", "gz"]].to_numpy(float)
    if pts.shape[0] == 0:
        return df_pts.iloc[0:0].copy()

    remaining = list(range(pts.shape[0]))
    selected = []
    rng = np.random.default_rng(seed)

    while remaining and len(selected) < target_n:
        pick_pos = int(rng.integers(0, len(remaining)))
        pick_idx = remaining[pick_pos]
        pick_pt = pts[pick_idx][None, :]

        rem_pts = pts[np.array(remaining)]
        nbrs = NearestNeighbors(radius=radius, algorithm="ball_tree")
        nbrs.fit(rem_pts)
        neigh_pos = nbrs.radius_neighbors(pick_pt, return_distance=False)[0].tolist()

        neigh_global = [remaining[p] for p in neigh_pos]
        selected.append(pick_idx)

        remove_set = set(neigh_global)
        remaining = [i for i in remaining if i not in remove_set]

    return df_pts.iloc[selected].reset_index(drop=True)

def process_one_cif(cif_path: str, out_root: str, df_disp: pd.DataFrame, chosen_angles: np.ndarray):
    rnum = extract_r_from_path(cif_path)
    disp_val_str = extract_t_from_path(cif_path)
    disp_val = float(disp_val_str)
    base_upper_angle = extract_angle_from_filename(cif_path)

    # Get structural displacement vector to set bounds dynamically
    disp_vec_parent = get_displacement_for_cif(cif_path, df_disp)
    if disp_vec_parent is not None:
        D_bilayer = np.linalg.norm(disp_vec_parent)
    else:
        D_bilayer = 7.58  # dataset mean fallback

    # Equal Separation bounds
    TM_MIN = D_bilayer - 0.5
    TM_MAX = D_bilayer + 0.5

    structure = read(cif_path)
    frac = structure.get_scaled_positions()
    structure.set_scaled_positions(adjust_fractional(frac))
    positions = structure.get_positions()
    cell = structure.get_cell()
    cell_arr = np.array(cell)
    pbc = structure.get_pbc()
    symbols = np.array(structure.get_chemical_symbols())

    df_atoms = pd.DataFrame(positions, columns=["x", "y", "z"])
    df_atoms.insert(0, "Element", symbols)

    carbon_df = df_atoms[df_atoms["Element"] == "C"].sort_values(by="z").reset_index(drop=True)
    hydrogen_df = df_atoms[df_atoms["Element"] == "H"].sort_values(by="z").reset_index(drop=True)
    oxygen_df = df_atoms[df_atoms["Element"] == "O"].sort_values(by="z").reset_index(drop=True)
    if len(carbon_df) < 2:
        return

    split_c = len(carbon_df) // 2
    lower_c = carbon_df.iloc[:split_c]
    upper_c = carbon_df.iloc[split_c:]

    split_h = len(hydrogen_df) // 2
    lower_h = hydrogen_df.iloc[:split_h]
    upper_h = hydrogen_df.iloc[split_h:]

    split_o = len(oxygen_df) // 2
    lower_o = oxygen_df.iloc[:split_o]
    upper_o = oxygen_df.iloc[split_o:]

    centroid_lower = np.concatenate((lower_c[["x","y","z"]].values, lower_o[["x","y","z"]].values))
    centroid_upper = np.concatenate((upper_c[["x","y","z"]].values, upper_o[["x","y","z"]].values))
    c_l = centroid_lower.mean(axis=0)
    c_u = centroid_upper.mean(axis=0)

    # ---------------- DYNAMIC ELEMENT-WISE THRESHOLD CALCULATION ----------------
    lower_pos = np.concatenate([lower_c[["x","y","z"]].values, 
                                lower_h[["x","y","z"]].values, 
                                lower_o[["x","y","z"]].values])
    lower_elems = np.array(['C']*len(lower_c) + ['H']*len(lower_h) + ['O']*len(lower_o))

    upper_pos = np.concatenate([upper_c[["x","y","z"]].values, 
                                upper_h[["x","y","z"]].values, 
                                upper_o[["x","y","z"]].values])
    upper_elems = np.array(['C']*len(upper_c) + ['H']*len(upper_h) + ['O']*len(upper_o))

    # Compute PBC-aware pairwise distances between parent layers
    inv_cell = np.linalg.inv(cell_arr)
    frac_l = lower_pos @ inv_cell
    frac_u = upper_pos @ inv_cell
    diff_frac = frac_l[:, None, :] - frac_u[None, :, :]
    for i in range(3):
        if pbc[i]:
            diff_frac[:, :, i] -= np.round(diff_frac[:, :, i])
    parent_dists = np.linalg.norm(diff_frac @ cell_arr, axis=2)
    d_min_parent = parent_dists.min()  # absolute closest atomic contact

    # Group and find minimum parent distance for each unique element pair
    unique_elements = ['C', 'H', 'O']
    d_min_parent_pairs = {}
    for el1 in unique_elements:
        for el2 in unique_elements:
            pair_key = tuple(sorted([el1, el2]))
            
            idx_l = np.where(lower_elems == el1)[0]
            idx_u = np.where(upper_elems == el2)[0]
            
            d_min = 99.0
            if len(idx_l) > 0 and len(idx_u) > 0:
                d_min = min(d_min, parent_dists[idx_l[:, None], idx_u[None, :]].min())
                
            idx_l2 = np.where(lower_elems == el2)[0]
            idx_u2 = np.where(upper_elems == el1)[0]
            if len(idx_l2) > 0 and len(idx_u2) > 0:
                d_min = min(d_min, parent_dists[idx_l2[:, None], idx_u2[None, :]].min())

            # Fallback to standard VDW sum scaled down slightly if pair contact not observed
            if d_min > 50.0:
                d_min = 0.7 * (VDW_RADII[el1] + VDW_RADII[el2])
                
            d_min_parent_pairs[pair_key] = d_min
    # -----------------------------------------------------------------------------

    coords = [(X_CART, y, z) for y in Y for z in Z]
    df_grid = pd.DataFrame(coords, columns=["x", "y", "z"])
    grid_xyz = df_grid[["x", "y", "z"]].to_numpy(float)

    d_l = np.linalg.norm(grid_xyz - c_l[None, :], axis=1)
    d_u = np.linalg.norm(grid_xyz - c_u[None, :], axis=1)

    df_vecs = pd.DataFrame({
        "gx": grid_xyz[:, 0], "gy": grid_xyz[:, 1], "gz": grid_xyz[:, 2],
        "d_l": d_l, "d_u": d_u
    })

    in_range_U = df_vecs["d_u"].between(TM_MIN, TM_MAX) & ~df_vecs["d_l"].between(0, TM_MIN)
    in_range_L = df_vecs["d_l"].between(TM_MIN, TM_MAX) & ~df_vecs["d_u"].between(0, TM_MIN)
    df_in_range = df_vecs.loc[in_range_U | in_range_L].copy()

    df_in_range = df_in_range.loc[df_in_range["d_l"] != df_in_range["d_u"]].copy()
    df_in_range["label"] = 0
    df_in_range.loc[df_in_range["d_u"] < df_in_range["d_l"], "label"] = 1

    df_lower_grids = df_in_range.loc[df_in_range["label"] == 0, ["gx","gy","gz"]].reset_index(drop=True)
    df_upper_grids = df_in_range.loc[df_in_range["label"] == 1, ["gx","gy","gz"]].reset_index(drop=True)

    df_lower_grids = select_grids_radius_prune(df_lower_grids, target_n=100, radius=4.0, seed=None)
    df_upper_grids = select_grids_radius_prune(df_upper_grids, target_n=100, radius=4.0, seed=None)

    df_all_sorted = df_atoms.sort_values(by="z").reset_index(drop=True)
    if len(df_all_sorted) != 156:
        return

    lower_chain = df_all_sorted.iloc[:78].copy().reset_index(drop=True)
    upper_chain = df_all_sorted.iloc[78:].copy().reset_index(drop=True)
    orig_all_elems = df_all_sorted["Element"].to_numpy()
    orig_all_xyz = df_all_sorted[["x","y","z"]].to_numpy(float)

    # Pre-build threshold matrix for bilayer-to-new-layer contact check
    threshold_matrix = np.zeros((len(orig_all_elems), 78))
    for i, e_f in enumerate(orig_all_elems):
        for j in range(78):
            template_e = lower_chain["Element"].iloc[j]
            pair_key = tuple(sorted([e_f, template_e]))
            threshold_matrix[i, j] = d_min_parent_pairs[pair_key]

    # Pre-convert data to arrays for JIT speed
    inv_cell = np.linalg.inv(cell_arr)

    def write_for_chain(kind: str, grid_idx: int, pivot: np.ndarray, origin_df: pd.DataFrame, neutral_angle_deg: float, full_elems: np.ndarray, full_xyz: np.ndarray):
        origin_xyz = origin_df[["x","y","z"]].to_numpy(float)

        new_chain = origin_df.copy()
        centroid_new = new_chain[["x","y","z"]].mean(axis=0).to_numpy()
        new_chain[["x","y","z"]] = new_chain[["x","y","z"]] + (pivot - centroid_new)

        new_xyz = new_chain[["x","y","z"]].to_numpy(float)
        new_elems = new_chain["Element"].to_numpy()

        neutral_xyz = rotate_about_x(new_xyz, neutral_angle_deg, pivot)

        disp_vec = np.array([disp_val, 0.0, 0.0])
        displaced_xyz = neutral_xyz + disp_vec[None, :]
        pivot_disp = pivot + disp_vec

        # Calculate shift direction
        ref_centroid = c_l if kind == "lower" else c_u
        body_vec = pivot_disp - ref_centroid
        u = body_vec / np.linalg.norm(body_vec)

        for ang in chosen_angles:
            rotated_xyz = rotate_about_x(displaced_xyz, -ang, pivot_disp)

            # Convert rotated_xyz to fractional space
            frac_orig = full_xyz @ inv_cell
            
            # ---------------- BISECTION OPTIMIZATION FOR EXACT GAP MATCHING ----------------
            # Find shift s to match d_min_parent exactly
            low, high = -2.5, 2.5
            for _ in range(12):
                mid = (low + high) / 2.0
                shifted_xyz = rotated_xyz + mid * u
                frac_new = shifted_xyz @ inv_cell
                d_mid = get_min_dist_jit(frac_orig, frac_new, cell_arr, pbc)
                if d_mid < d_min_parent:
                    low = mid
                else:
                    high = mid
            s_opt = (low + high) / 2.0
            optimized_xyz = rotated_xyz + s_opt * u
            # --------------------------------------------------------------------------------

            # PBC-aware JIT clash check using the dynamic element-wise threshold matrix
            frac_opt = optimized_xyz @ inv_cell
            if check_clash_pbc_jit(frac_orig, frac_opt, cell_arr, pbc, threshold_matrix):
                continue

            merged_elems = np.concatenate([full_elems, new_elems])
            merged_xyz = np.vstack([full_xyz, optimized_xyz])

            # Create Atoms object (DO NOT wrap coordinates to keep bonds unified and contiguous)
            atoms_out = Atoms(symbols=merged_elems, positions=merged_xyz, cell=cell, pbc=True)

            # ================= DYNAMIC CELL ENLARGEMENT =================
            orig_cell_arr = atoms_out.get_cell().array
            enlarged_cell = orig_cell_arr.copy()

            # Add 50 Å to the current b-vector length (keep direction)
            b_vec = enlarged_cell[1]
            b_len = np.linalg.norm(b_vec)
            if b_len > 1e-8:
                new_b_len = b_len + 50.0
                enlarged_cell[1] = b_vec * (new_b_len / b_len)
            else:
                enlarged_cell[1] = np.array([0.0, 50.0, 0.0])

            # Add 50 Å to the current c-vector length (keep direction)
            c_vec = enlarged_cell[2]
            c_len = np.linalg.norm(c_vec)
            if c_len > 1e-8:
                new_c_len = c_len + 50.0
                enlarged_cell[2] = c_vec * (new_c_len / c_len)
            else:
                enlarged_cell[2] = np.array([0.0, 0.0, 50.0])

            # Apply expanded cell without scaling or moving any coordinates
            atoms_out.set_cell(enlarged_cell, scale_atoms=False)
            # ============================================================

            out_dir = os.path.join(
                out_root,
                kind,
                f"r{rnum}",
                f"t{disp_val_str}"
            )
            os.makedirs(out_dir, exist_ok=True)
            out_name = f"t{disp_val_str}_{ang}_grid{grid_idx}.cif"
            atoms_out.write(os.path.join(out_dir, out_name))

    base_lower_angle = -float(rnum)
    for gi, row in df_lower_grids.iterrows():
        pivot = row.to_numpy(float)
        write_for_chain("lower", gi, pivot, lower_chain, base_lower_angle, orig_all_elems, orig_all_xyz)

    for gi, row in df_upper_grids.iterrows():
        pivot = row.to_numpy(float)
        write_for_chain("upper", gi, pivot, upper_chain, base_upper_angle, orig_all_elems, orig_all_xyz)

# ================= MULTIPROCESSING TASK WRAPPER =================
def process_one_cif_wrapper(task):
    """
    Worker task wrapper that processes a single CIF file.
    All input parameters are unpacked here.
    """
    cif_path, out_root, df_disp, chosen_angles = task
    try:
        process_one_cif(cif_path, out_root, df_disp, chosen_angles)
    except Exception as e:
        print(f"Error processing {cif_path}: {e}")

# ================= MAIN RUNNER =================
def main():
    import argparse
    parser = argparse.ArgumentParser(description="High-performance Trilayer CIF Generator")
    parser.add_argument("--input", default=ROOT_IN, help="Path to input cif files")
    parser.add_argument("--displacements", default=TM_CSV, help="Path to displacements CSV")
    parser.add_argument("--output", default=OUT_ROOT, help="Path to save output cifs")
    parser.add_argument("--cores", type=int, default=os.cpu_count(), help="Number of CPU cores to use")
    
    import sys
    # Bypass Jupyter/ipykernel args to avoid argument parser crash in notebooks
    if any("ipykernel" in arg or "jupyter" in arg for arg in sys.argv) or "ipykernel" in sys.modules:
        args = parser.parse_args([])
    else:
        args = parser.parse_args()

    # Load displacements mapping
    if not os.path.exists(args.displacements):
        print(f"Warning: Displacements CSV not found at {args.displacements}. Will fall back to default D_bilayer = 7.58 Å.")
        df_disp = pd.DataFrame(columns=["file_path", "X", "Y", "Z"])
    else:
        df_disp = pd.read_csv(args.displacements)

    all_cifs = []
    for root, _, files in os.walk(args.input):
        for f in files:
            if f.lower().endswith(".cif"):
                all_cifs.append(os.path.join(root, f))
    
    print(f"Total CIF files found: {len(all_cifs)}")
    print(f"Initializing multi-threading with {args.cores} worker threads...")

    # Set up random choice of angles
    rng = np.random.default_rng(42)
    chosen_angles = rng.choice(ANGLES, size=5, replace=False)
    print(f"Randomly chosen rotation angles for this execution run: {chosen_angles}")

    # Prepare parallel tasks
    tasks = [(cif_path, args.output, df_disp, chosen_angles) for cif_path in all_cifs]

    # Run ThreadPoolExecutor (highly efficient in parallel because Numba releases the GIL with nogil=True)
    from concurrent.futures import ThreadPoolExecutor, as_completed
    
    with ThreadPoolExecutor(max_workers=args.cores) as executor:
        futures = {executor.submit(process_one_cif_wrapper, t): t[0] for t in tasks}
        
        # tqdm progress bar
        for fut in tqdm(as_completed(futures), total=len(futures), desc="Generating Trilayers in Parallel"):
            fut.result()

    print("All trilayers generated successfully.")

if __name__ == "__main__":
    main()


Total CIF files found: 300
Initializing multi-threading with 8 worker threads...
Randomly chosen rotation angles for this execution run: [220 320 280  80 340]


Generating Trilayers in Parallel: 100%|██████████| 300/300 [01:20<00:00,  3.73it/s]

All trilayers generated successfully.


### Features

In [7]:
from __future__ import annotations

import os
import re
import sys
import numpy as np
import pandas as pd
from tqdm import tqdm
from ase.io import read
from pathlib import Path
from scipy.spatial import cKDTree as KDTree
from concurrent.futures import ThreadPoolExecutor, as_completed

# ============================================
# ===== PORTABLE PATH & CONFIG SETTINGS =====
# ============================================
try:
    SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:
    # Jupyter Notebook fallback
    SCRIPT_DIR = Path.cwd()

# Automatically resolve paths relative to the project root
PROJECT_DIR = SCRIPT_DIR if (SCRIPT_DIR / "results").exists() else SCRIPT_DIR.parent
BASE_DIR    = PROJECT_DIR / "results" / "3_layer"
OUTPUT_DIR  = PROJECT_DIR / "results"
OUTPUT_CSV  = "trilayer_features.csv"

# van der Waals radii (Å) and tolerance
rvw_H   = 1.20
rvw_O   = 1.53
rvw_err = 0.10

# ------------------------------------------------
def validate_structure(atoms):
    if len(atoms) == 0:
        raise ValueError("Structure has zero atoms.")
    if atoms.get_cell().volume <= 0:
        raise ValueError("Invalid or zero-volume cell.")
    if np.isnan(atoms.get_positions()).any():
        raise ValueError("NaN detected in atomic positions.")

# ---------- geometry helpers ----------
def wrap_positions_custom(positions, cell, pbc, center=(0.5, 0.5, 0.5)):
    """
    Consistent coordinate wrapping centered at 0.5 to keep layers 
    from splitting across cell boundaries during clustering.
    """
    inv_cell = np.linalg.inv(cell)
    frac = np.dot(positions, inv_cell)
    for i in range(3):
        if pbc[i]:
            shift = frac[:, i] - center[i] + 0.5
            frac[:, i] = (shift % 1.0) + center[i] - 0.5
    wrapped_pos = np.dot(frac, cell)
    return wrapped_pos

def load_and_unwrap_atoms(cif_path: str):
    """
    Read CIF -> unwrap positions consistently to avoid boundary splits.
    Sets PBC to False temporarily for safe non-periodic KDTree search.
    """
    atoms = read(cif_path)
    validate_structure(atoms)

    pos = atoms.get_positions()
    cell = atoms.get_cell().array
    pbc  = atoms.get_pbc()

    # Wrap around center to prevent boundary layer-splitting
    unwrapped = wrap_positions_custom(pos, cell, pbc, center=(0.5, 0.5, 0.5))
    atoms.set_positions(unwrapped)

    # Disable PBC for KDTree distance calculations
    atoms.pbc = False
    return atoms

# ---------- feature helpers ----------
def mean_or_default(values, default=3.0) -> float:
    return float(np.mean(values)) if len(values) else float(default)

def coords_of_df(df: pd.DataFrame) -> np.ndarray:
    if df.empty:
        return np.zeros((0, 3), dtype=float)
    return df[["x", "y", "z"]].to_numpy(dtype=float)

def distances_within(lower_coords: np.ndarray, upper_coords: np.ndarray, cutoff: float):
    if lower_coords.shape[0] == 0 or upper_coords.shape[0] == 0:
        return []

    tree = KDTree(lower_coords)
    dists, _ = tree.query(upper_coords, k=1)

    dists = np.asarray(dists).ravel()
    kept = dists[dists <= cutoff]
    return kept.tolist()

# ---------- per-file feature computation ----------
def compute_bounds_for_trilayer(cif_path: str) -> dict:
    try:
        atoms = load_and_unwrap_atoms(cif_path)

        symbols   = atoms.get_chemical_symbols()
        positions = atoms.get_positions()
        df = pd.DataFrame(positions, columns=["x", "y", "z"])
        df.insert(0, "Element", symbols)

        # Sort all 234 atoms by their wrapped Z-coordinate to identify layers
        df_sorted = df.sort_values(by="z").reset_index(drop=True)
        if len(df_sorted) != 234:
            raise ValueError(f"Expected 234 atoms in trilayer structure, found {len(df_sorted)}")

        # Split into Lower, Middle, and Upper layers (78 atoms each)
        layer_lower  = df_sorted.iloc[:78].copy()
        layer_middle = df_sorted.iloc[78:156].copy()
        layer_upper  = df_sorted.iloc[156:].copy()

        # Extract elements coordinates for each layer
        lH_c = coords_of_df(layer_lower[layer_lower["Element"] == "H"])
        lO_c = coords_of_df(layer_lower[layer_lower["Element"] == "O"])

        mH_c = coords_of_df(layer_middle[layer_middle["Element"] == "H"])
        mO_c = coords_of_df(layer_middle[layer_middle["Element"] == "O"])

        uH_c = coords_of_df(layer_upper[layer_upper["Element"] == "H"])
        uO_c = coords_of_df(layer_upper[layer_upper["Element"] == "O"])

        # Cutoffs
        cut_HH = 2 * rvw_H + rvw_err
        cut_OO = 2 * rvw_O + rvw_err
        cut_OH = rvw_O + rvw_H + rvw_err

        # =======================================================
        # 1. Lower-Middle (L_M) Interface Features
        # =======================================================
        dist_HH_LM = distances_within(lH_c, mH_c, cut_HH)
        dist_OO_LM = distances_within(lO_c, mO_c, cut_OO)
        dist_OH_LM = distances_within(lH_c, mO_c, cut_OH)  # middle O to lower H
        dist_HO_LM = distances_within(lO_c, mH_c, cut_OH)  # middle H to lower O

        # =======================================================
        # 2. Middle-Upper (M_U) Interface Features
        # =======================================================
        dist_HH_MU = distances_within(mH_c, uH_c, cut_HH)
        dist_OO_MU = distances_within(mO_c, uO_c, cut_OO)
        dist_OH_MU = distances_within(mH_c, uO_c, cut_OH)  # upper O to middle H
        dist_HO_MU = distances_within(mO_c, uH_c, cut_OH)  # upper H to middle O

        return {
            "status": "success",
            "file_path": cif_path,
            
            # Lower-Middle (L_M) Features
            "L_M_avg_HH_dist": mean_or_default(dist_HH_LM),
            "L_M_avg_OO_dist": mean_or_default(dist_OO_LM),
            "L_M_avg_OH_dist": mean_or_default(dist_OH_LM),
            "L_M_avg_HO_dist": mean_or_default(dist_HO_LM),
            "L_M_count_HH": len(dist_HH_LM),
            "L_M_count_OO": len(dist_OO_LM),
            "L_M_count_OH": len(dist_OH_LM),
            "L_M_count_HO": len(dist_HO_LM),
            "L_M_sum_of_count": len(dist_HH_LM) + len(dist_OO_LM) + len(dist_OH_LM) + len(dist_HO_LM),
            
            # Middle-Upper (M_U) Features
            "M_U_avg_HH_dist": mean_or_default(dist_HH_MU),
            "M_U_avg_OO_dist": mean_or_default(dist_OO_MU),
            "M_U_avg_OH_dist": mean_or_default(dist_OH_MU),
            "M_U_avg_HO_dist": mean_or_default(dist_HO_MU),
            "M_U_count_HH": len(dist_HH_MU),
            "M_U_count_OO": len(dist_OO_MU),
            "M_U_count_OH": len(dist_OH_MU),
            "M_U_count_HO": len(dist_HO_MU),
            "M_U_sum_of_count": len(dist_HH_MU) + len(dist_OO_MU) + len(dist_OH_MU) + len(dist_HO_MU),
        }

    except Exception as e:
        return {"status": "error", "file_path": cif_path, "error": str(e)}

# ---------- sorting path parser ----------
def path_sort_key(path):
    """
    Parses paths like 'results/3_layer/lower/r20/t1.2/t1.2_80_grid15.cif'
    into sorting keys (kind, rotation, displacement, angle, grid).
    """
    path_str = str(path).replace('\\', '/')
    kind = "lower" if "/lower/" in path_str else "upper"
    
    rot_match = re.search(r'/r(\d+)/', path_str)
    rot_val = int(rot_match.group(1)) if rot_match else 0

    disp_match = re.search(r'/t(\d+(?:\.\d+)?)/', path_str)
    disp_val = float(disp_match.group(1)) if disp_match else 0.0

    fname = os.path.basename(path_str)
    angle_match = re.search(r'_(\d+)_grid(\d+)\.cif$', fname)
    if angle_match:
        angle_val = int(angle_match.group(1))
        grid_val = int(angle_match.group(2))
    else:
        angle_val = 0
        grid_val = 0

    return (kind, rot_val, disp_val, angle_val, grid_val)

# ------------------ MAIN EXECUTION ------------------
def main():
    if not BASE_DIR.exists():
        print(f"Error: Target directory does not exist: {BASE_DIR}")
        return

    # Gather files
    cif_paths = []
    for root, _, files in os.walk(BASE_DIR):
        for file in files:
            if file.lower().endswith(".cif"):
                cif_paths.append(os.path.join(root, file))

    total_files = len(cif_paths)
    print(f"Found {total_files} trilayer CIF files to extract features from.")

    # Use ThreadPoolExecutor for notebook safety
    num_workers = max(1, os.cpu_count() - 1)
    print(f"Processing using {num_workers} parallel workers (ThreadPoolExecutor)...")

    all_results = []
    errors = []

    with ThreadPoolExecutor(max_workers=num_workers) as executor:
        futures = {executor.submit(compute_bounds_for_trilayer, p): p for p in cif_paths}
        
        with tqdm(total=total_files, desc="Extracting trilayer features", unit="file") as pbar:
            for future in as_completed(futures):
                res = future.result()
                if res["status"] == "success":
                    all_results.append(res)
                else:
                    errors.append((res["file_path"], res["error"]))
                pbar.update(1)

    # Convert to DataFrame
    df_all = pd.DataFrame(all_results)
    
    if df_all.empty:
        print("No features extracted.")
        return

    # Clean up status column
    df_all.drop(columns=["status"], errors="ignore")

    # Sort results structurally
    df_all["sort_key"] = df_all["file_path"].apply(path_sort_key)
    df_all = df_all.sort_values(by="sort_key").drop(columns=["sort_key"]).reset_index(drop=True)

    # Ensure output columns order
    cols_feature = [c for c in df_all.columns if c not in ("file_path", "status")]
    cols_to_keep = ["file_path"] + cols_feature
    df_final = df_all[cols_to_keep]

    # Save to CSV
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    out_path = OUTPUT_DIR / OUTPUT_CSV
    df_final.to_csv(out_path, index=False)
    print(f"\nFeature extraction complete! Saved to: {out_path}")

    # Display sample
    with pd.option_context("display.max_columns", None, "display.width", 160):
        print(df_final.head(10))

    if errors:
        print(f"\n--- Errors encountered during extraction ({len(errors)} total) ---")
        for path, msg in errors[:20]:
            print(f"- {path}: {msg}")


if __name__ == "__main__":
    main()


Found 222 trilayer CIF files to extract features from.
Processing using 7 parallel workers (ThreadPoolExecutor)...


Extracting trilayer features: 100%|██████████| 222/222 [00:31<00:00,  7.13file/s]


Feature extraction complete! Saved to: d:\New folder\project\results\trilayer_features.csv
                                           file_path  L_M_avg_HH_dist  L_M_avg_OO_dist  L_M_avg_OH_dist  L_M_avg_HO_dist  L_M_count_HH  L_M_count_OO  \
0  d:\New folder\project\results\3_layer\lower\r0...         2.382907         2.611873         2.256996         2.392568             7             5   
1  d:\New folder\project\results\3_layer\lower\r0...         2.351724         2.768675         1.800385         2.395798            10            10   
2  d:\New folder\project\results\3_layer\lower\r0...         2.336775         2.946815         2.324762         2.382820             7             8   
3  d:\New folder\project\results\3_layer\lower\r0...         2.290459         2.626194         2.291629         2.333699             7             6   
4  d:\New folder\project\results\3_layer\lower\r0...         2.370821         2.610325         2.263696         2.390548             6             5

### Model Inference and saving

In [8]:
from __future__ import annotations

import os
import re
import json
import shutil
import warnings
from pathlib import Path
from typing import Dict, Any, Union

import numpy as np
import pandas as pd

# Suppress sklearn/onnx warnings
warnings.filterwarnings("ignore")

# ============================================
# ===== PORTABLE PATH & CONFIG SETTINGS =====
# ============================================
try:
    SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:
    # Jupyter Notebook fallback
    SCRIPT_DIR = Path.cwd()

# Automatically resolve paths relative to the project root
PROJECT_DIR = SCRIPT_DIR if (SCRIPT_DIR / "results").exists() else SCRIPT_DIR.parent
MODEL_DIR   = PROJECT_DIR / "results"
DATA_PATH   = MODEL_DIR / "trilayer_features.csv"
JOBLIB_PATH = MODEL_DIR / "pipeline_model_calibrated.joblib"
ONNX_PATH   = MODEL_DIR / "pipeline_model_calibrated.onnx"
METADATA_PATH = MODEL_DIR / "model_threshold.json"
BASE_DIR    = MODEL_DIR / "3_layer"  # Original source trilayers directory

# Destination folder for stable (Class 0) trilayer structures
DEST_DIR    = PROJECT_DIR / "negative_3_cifs"

# ================= ONNX FALLBACK HELPERS =================
def _ensure_float32(X: np.ndarray) -> np.ndarray:
    if not isinstance(X, np.ndarray):
        X = np.asarray(X)
    if X.dtype != np.float32:
        X = X.astype(np.float32)
    return X

def _extract_pos_class_proba(ort_outputs: Any) -> np.ndarray:
    probs = ort_outputs[1]
    if isinstance(probs, list):
        if len(probs) == 0:
            return np.array([], dtype=np.float32)
        if isinstance(probs[0], dict):
            return np.array([float(d.get(1, 0.0)) for d in probs], dtype=np.float32)
    probs = np.asarray(probs)
    if probs.ndim == 2 and probs.shape[1] >= 2:
        return probs[:, 1].astype(np.float32)
    raise ValueError(f"Unsupported ONNX probability output format: shape={getattr(probs, 'shape', None)}")

def run_predictions():
    print(f"Project directory: {PROJECT_DIR}")
    print(f"Loading dataset from: {DATA_PATH}")
    if not os.path.exists(DATA_PATH):
        raise FileNotFoundError(f"Trilayer features not found at {DATA_PATH}. Please run feature extraction first.")

    df_full = pd.read_csv(DATA_PATH)

    # 1) Load Config & Threshold
    if not os.path.exists(METADATA_PATH):
        raise FileNotFoundError(f"Model threshold config not found at {METADATA_PATH}")

    with open(METADATA_PATH, "r") as f:
        config = json.load(f)

    threshold = float(config["threshold"])
    feature_names = list(config["features"])

    # 2) Extract Interface Features dynamically matching the model configuration
    lm_cols = [f"L_M_{col}" for col in feature_names]
    mu_cols = [f"M_U_{col}" for col in feature_names]

    for col in lm_cols + mu_cols:
        if col not in df_full.columns:
            raise ValueError(f"Feature table is missing required interface column: {col}")

    X_lm = df_full[lm_cols].copy()
    X_lm.columns = feature_names

    X_mu = df_full[mu_cols].copy()
    X_mu.columns = feature_names

    # 3) Model Inference (Supports Joblib Pipeline and ONNX fallback)
    probs_lm = None
    probs_mu = None

    # Try Joblib first (contains StandardScaler + CalibratedClassifierCV)
    if os.path.exists(JOBLIB_PATH):
        try:
            import joblib
            model = joblib.load(JOBLIB_PATH)
            probs_lm = model.predict_proba(X_lm)[:, 1]
            probs_mu = model.predict_proba(X_mu)[:, 1]
        except Exception as e:
            print(f"Warning: Joblib prediction failed ({e}). Attempting ONNX fallback...")

    # Try ONNX fallback if joblib is not available or failed
    if probs_lm is None or probs_mu is None:
        if not os.path.exists(ONNX_PATH):
            raise FileNotFoundError("Both Joblib and ONNX model files are missing.")
        
        import onnxruntime as ort
        sess = ort.InferenceSession(str(ONNX_PATH), providers=["CPUExecutionProvider"])
        input_name = sess.get_inputs()[0].name
        
        X_lm_arr = _ensure_float32(X_lm.to_numpy())
        X_mu_arr = _ensure_float32(X_mu.to_numpy())
        
        ort_lm = sess.run(None, {input_name: X_lm_arr})
        ort_mu = sess.run(None, {input_name: X_mu_arr})
        
        probs_lm = _extract_pos_class_proba(ort_lm)
        probs_mu = _extract_pos_class_proba(ort_mu)

    # 4) Determine Stability Classes
    # Class 0 (Stable) if Unstable probability is strictly less than threshold
    preds_lm = (probs_lm >= threshold).astype(int)
    preds_mu = (probs_mu >= threshold).astype(int)

    # A trilayer structure is stable if BOTH interfaces are classified as stable (Class 0)
    is_stable_trilayer = (preds_lm == 0) & (preds_mu == 0)
    final_preds = np.where(is_stable_trilayer, 0, 1)

    # 5) Analyze class distribution predictions
    pred_counts = pd.Series(final_preds).value_counts().to_dict()
    lm_counts = pd.Series(preds_lm).value_counts().to_dict()
    mu_counts = pd.Series(preds_mu).value_counts().to_dict()

    print("\n================ PREDICTION DISTRIBUTION ================")
    print(f"Total dataset size: {len(df_full)} structures")
    print(f"Predicted L_M interface Class 0 (Stable): {lm_counts.get(0, 0)} | Class 1 (Unstable): {lm_counts.get(1, 0)}")
    print(f"Predicted M_U interface Class 0 (Stable): {mu_counts.get(0, 0)} | Class 1 (Unstable): {mu_counts.get(1, 0)}")
    print(f"Overall Trilayer Class 0 (Stable)       : {pred_counts.get(0, 0)} | Class 1 (Unstable): {pred_counts.get(1, 0)}")
    print("=========================================================")

    # 6) Identify indices of predicted Class 0 (Stable) trilayers
    class0_indices = np.where(final_preds == 0)[0]
    total_class0 = len(class0_indices)

    # Clean destination directory if it exists to ensure freshness
    if os.path.exists(DEST_DIR):
        print(f"\nRemoving existing destination folder: {DEST_DIR}")
        shutil.rmtree(DEST_DIR)
    os.makedirs(DEST_DIR, exist_ok=True)

    copied_count = 0
    missing_count = 0

    print("\nCopying stable structures...")
    for idx in class0_indices:
        orig_path = df_full.loc[idx, "file_path"]
        
        # Get relative path relative to 3_layer directory
        rel_path = os.path.relpath(orig_path, BASE_DIR)
        if rel_path.startswith(".."):
            parts = orig_path.replace("\\", "/").split("/3_layer/")
            if len(parts) > 1:
                rel_path = parts[-1]
            else:
                rel_path = os.path.basename(orig_path)
        
        source_file = os.path.join(BASE_DIR, rel_path)
        dest_file = os.path.join(DEST_DIR, rel_path)
        
        if os.path.exists(source_file):
            os.makedirs(os.path.dirname(dest_file), exist_ok=True)
            shutil.copy2(source_file, dest_file)
            copied_count += 1
        else:
            print(f"Warning: Source file not found: {source_file}")
            missing_count += 1

    print("\n================ COPY SUMMARY ================")
    print(f"Total Predicted Class 0 (Stable): {total_class0}")
    print(f"Successfully copied             : {copied_count} files")
    print(f"Missing source files            : {missing_count}")
    print(f"Saved into                      : {DEST_DIR}")
    print("==============================================")

if __name__ == "__main__":
    run_predictions()


Project directory: d:\New folder\project
Loading dataset from: d:\New folder\project\results\trilayer_features.csv

================ PREDICTION DISTRIBUTION ================
Total dataset size: 221 structures
Predicted L_M interface Class 0 (Stable): 215 | Class 1 (Unstable): 6
Predicted M_U interface Class 0 (Stable): 160 | Class 1 (Unstable): 61
Overall Trilayer Class 0 (Stable)       : 155 | Class 1 (Unstable): 66

Copying stable structures...

================ COPY SUMMARY ================
Total Predicted Class 0 (Stable): 155
Successfully copied             : 155 files
Missing source files            : 0
Saved into                      : d:\New folder\project\negative_3_cifs


In [2]:
import pandas as pd
trial = pd.read_csv("D:/New folder/project/results/negative_energy_geometry_distances.csv")

In [3]:
trial.head()

,file_id,file_path,adjusted_energy,all_atoms_distance_mic,carbon_distance_mic,displacement_distance,X,Y,Z
0,r0_t0_t0_240.cif,d:\New folder\project\data\r0\t0\t0_240.cif,-0.00117,7.262548,7.729099,7.729099,2.626428,-0.908362,7.212193
1,r0_t1.2_t1.2_240.cif,d:\New folder\project\data\r0\t1.2\t1.2_240.cif,-0.01050,7.050208,7.359408,7.359408,1.128858,-0.854793,7.221904
2,r0_t1.2_t1.2_260.cif,d:\New folder\project\data\r0\t1.2\t1.2_260.cif,-0.05394,6.638044,6.907802,6.907802,0.901665,-0.516722,6.829182
3,r0_t2.4_t2.4_240.cif,d:\New folder\project\data\r0\t2.4\t2.4_240.cif,-0.08396,7.117104,7.405533,7.405533,0.336812,-1.293310,7.283943
4,r0_t2.4_t2.4_260.cif,d:\New folder\project\data\r0\t2.4\t2.4_260.cif,-0.14515,6.634964,6.885281,6.885281,0.529677,-0.439607,6.850787


In [ ]:
# drop all_atoms_distance_mic, carbon_distance_mic from trial and save it
trial = trial.drop(columns=['all_atoms_distance_mic', 'carbon_distance_mic'])

trial.head()

,file_id,file_path,adjusted_energy,displacement_distance,X,Y,Z
0,r0_t0_t0_240.cif,d:\New folder\project\data\r0\t0\t0_240.cif,-0.00117,7.729099,2.626428,-0.908362,7.212193
1,r0_t1.2_t1.2_240.cif,d:\New folder\project\data\r0\t1.2\t1.2_240.cif,-0.01050,7.359408,1.128858,-0.854793,7.221904
2,r0_t1.2_t1.2_260.cif,d:\New folder\project\data\r0\t1.2\t1.2_260.cif,-0.05394,6.907802,0.901665,-0.516722,6.829182
3,r0_t2.4_t2.4_240.cif,d:\New folder\project\data\r0\t2.4\t2.4_240.cif,-0.08396,7.405533,0.336812,-1.293310,7.283943
4,r0_t2.4_t2.4_260.cif,d:\New folder\project\data\r0\t2.4\t2.4_260.cif,-0.14515,6.885281,0.529677,-0.439607,6.850787


In [6]:
trial = trial.rename(columns={'displacement_distance': 'geometric_center_distance'})
trial.head()

,file_id,file_path,adjusted_energy,geometric_center_distance,X,Y,Z
0,r0_t0_t0_240.cif,d:\New folder\project\data\r0\t0\t0_240.cif,-0.00117,7.729099,2.626428,-0.908362,7.212193
1,r0_t1.2_t1.2_240.cif,d:\New folder\project\data\r0\t1.2\t1.2_240.cif,-0.01050,7.359408,1.128858,-0.854793,7.221904
2,r0_t1.2_t1.2_260.cif,d:\New folder\project\data\r0\t1.2\t1.2_260.cif,-0.05394,6.907802,0.901665,-0.516722,6.829182
3,r0_t2.4_t2.4_240.cif,d:\New folder\project\data\r0\t2.4\t2.4_240.cif,-0.08396,7.405533,0.336812,-1.293310,7.283943
4,r0_t2.4_t2.4_260.cif,d:\New folder\project\data\r0\t2.4\t2.4_260.cif,-0.14515,6.885281,0.529677,-0.439607,6.850787


In [9]:
import os
folder_path = "D:/New folder/project/results"  # Mac/Linux format
full_output_path = os.path.join(folder_path, "geometric_center.csv")
trial.to_csv(full_output_path, index=False)